In [12]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [13]:
stocks = [
    "ADANIENT.NS",
    "ADANIPORTS.NS",
    "APOLLOHOSP.NS",
    "ASIANPAINT.NS",
    "AXISBANK.NS",
    "BAJAJ-AUTO.NS",
    "BAJFINANCE.NS",
    "BEL.NS",
    "BHARTIARTL.NS",
    "CIPLA.NS",
    "COALINDIA.NS",
    "DRREDDY.NS",
    "EICHERMOT.NS",
    "ETERNAL.NS",   # formerly Zomato
    "GRASIM.NS",
    "HCLTECH.NS",
    "HDFCBANK.NS",
    "HDFCLIFE.NS",
    "HEROMOTOCO.NS",
    "HINDALCO.NS",
    "HINDUNILVR.NS",
    "ICICIBANK.NS",
    "INDUSINDBK.NS",
    "INFY.NS",
    "ITC.NS",
    "JIOFIN.NS",
    "JSWSTEEL.NS",
    "KOTAKBANK.NS",
    "LT.NS",
    "M&M.NS",
    "MARUTI.NS",
    "NESTLEIND.NS",
    "NTPC.NS",
    "ONGC.NS",
    "POWERGRID.NS",
    "RELIANCE.NS",
    "SBILIFE.NS",
    "SBIN.NS",
    "SHRIRAMFIN.NS",
    "SUNPHARMA.NS",
    "TATACONSUM.NS",
    "TATAMOTORS.NS",
    "TATASTEEL.NS",
    "TCS.NS",
    "TECHM.NS",
    "TITAN.NS",
    "TRENT.NS",
    "ULTRACEMCO.NS",
    "WIPRO.NS"
]


start = "2020-01-01"

In [14]:
def backtest_donchian(df):

    df = df.copy()

    df["don_high"] = (
        df["Close"]
        .rolling(20)
        .max()
        .shift(1)
    )

    df["don_low"] = (
        df["Close"]
        .rolling(10)
        .min()
        .shift(1)
    )

    position = 0
    positions = []

    for i in range(len(df)):

        if position == 0:

            if (
                df["Close"].iloc[i]
                >
                df["don_high"].iloc[i]
            ):
                position = 1

        else:

            if (
                df["Close"].iloc[i]
                <
                df["don_low"].iloc[i]
            ):
                position = 0

        positions.append(position)

    df["position"] = positions

    df["ret"] = np.log(df["Close"]).diff()

    df["strategy_ret"] = (
        df["position"]
        .shift(1)
        *
        df["ret"]
    )

    strategy = np.exp(
        df["strategy_ret"]
        .fillna(0)
        .cumsum()
    )

    buyhold = np.exp(
        df["ret"]
        .fillna(0)
        .cumsum()
    )

    return strategy, buyhold

In [15]:
results = []

for stock in stocks:

    try:

        df = yf.download(
            stock,
            start="2020-01-01",
            auto_adjust=True,
            progress=False
        )

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        strategy, buyhold = backtest_donchian(df)

        strat_return = (
            strategy.iloc[-1] - 1
        ) * 100

        bh_return = (
            buyhold.iloc[-1] - 1
        ) * 100

        results.append([
            stock,
            round(strat_return,2),
            round(bh_return,2)
        ])

        print(f"Done: {stock}")

    except Exception as e:

        print(stock, e)

Done: ADANIENT.NS
Done: ADANIPORTS.NS
Done: APOLLOHOSP.NS
Done: ASIANPAINT.NS
Done: AXISBANK.NS
Done: BAJAJ-AUTO.NS
Done: BAJFINANCE.NS
Done: BEL.NS
Done: BHARTIARTL.NS
Done: CIPLA.NS
Done: COALINDIA.NS
Done: DRREDDY.NS
Done: EICHERMOT.NS
Done: ETERNAL.NS
Done: GRASIM.NS
Done: HCLTECH.NS
Done: HDFCBANK.NS
Done: HDFCLIFE.NS
Done: HEROMOTOCO.NS
Done: HINDALCO.NS
Done: HINDUNILVR.NS
Done: ICICIBANK.NS
Done: INDUSINDBK.NS
Done: INFY.NS
Done: ITC.NS
Done: JIOFIN.NS
Done: JSWSTEEL.NS
Done: KOTAKBANK.NS
Done: LT.NS
Done: M&M.NS
Done: MARUTI.NS
Done: NESTLEIND.NS
Done: NTPC.NS
Done: ONGC.NS
Done: POWERGRID.NS
Done: RELIANCE.NS
Done: SBILIFE.NS
Done: SBIN.NS
Done: SHRIRAMFIN.NS
Done: SUNPHARMA.NS
Done: TATACONSUM.NS


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


TATAMOTORS.NS single positional indexer is out-of-bounds
Done: TATASTEEL.NS
Done: TCS.NS
Done: TECHM.NS
Done: TITAN.NS
Done: TRENT.NS
Done: ULTRACEMCO.NS
Done: WIPRO.NS


In [16]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Stock",
        "Strategy Return %",
        "Buy & Hold %"
    ]
)

results_df["Alpha"] = (
    results_df["Strategy Return %"]
    -
    results_df["Buy & Hold %"]
)

results_df.sort_values(
    "Alpha",
    ascending=False,
    inplace=True
)

results_df

,Stock,Strategy Return %,Buy & Hold %,Alpha
6,BAJFINANCE.NS,260.30,132.58,127.72
47,WIPRO.NS,142.98,59.00,83.98
22,INDUSINDBK.NS,21.94,-33.90,55.84
17,HDFCLIFE.NS,25.21,-2.96,28.17
42,TCS.NS,39.15,15.42,23.73
25,JIOFIN.NS,16.94,-1.63,18.57
23,INFY.NS,75.43,69.65,5.78
15,HCLTECH.NS,153.81,151.10,2.71
3,ASIANPAINT.NS,31.83,60.25,-28.42
20,HINDUNILVR.NS,-4.85,25.41,-30.26


In [17]:
print("Average Strategy Return:",
      round(
          results_df["Strategy Return %"].mean(),
          2
      ),
      "%")

print("Average Buy Hold Return:",
      round(
          results_df["Buy & Hold %"].mean(),
          2
      ),
      "%")

print("Stocks Beating Buy Hold:",
      (
          results_df["Alpha"] > 0
      ).sum(),
      "/",
      len(results_df))

Average Strategy Return: 91.65 %
Average Buy Hold Return: 251.51 %
Stocks Beating Buy Hold: 8 / 48
